In [4]:
import pandas as pd
cols_needed = [
    'SQLDATE', 'MonthYear', 'Year',
    'Actor1CountryCode', 'Actor2CountryCode',
    'EventCode', 'QuadClass', 'GoldsteinScale',
    'NumMentions', 'NumSources', 'NumArticles',
    'dyad'
]
df = pd.read_csv('../data/processed/gdelt_dyad.csv', usecols=cols_needed)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3502782 entries, 0 to 3502781
Data columns (total 12 columns):
 #   Column             Dtype  
---  ------             -----  
 0   SQLDATE            int64  
 1   MonthYear          int64  
 2   Year               int64  
 3   Actor1CountryCode  str    
 4   Actor2CountryCode  str    
 5   EventCode          int64  
 6   QuadClass          int64  
 7   GoldsteinScale     float64
 8   NumMentions        int64  
 9   NumSources         int64  
 10  NumArticles        int64  
 11  dyad               str    
dtypes: float64(1), int64(8), str(3)
memory usage: 364.1 MB


In [ ]:
# y label define
import numpy as np

def classify_event(df):
    df_copy = df.copy()
    goldstein = df_copy['GoldsteinScale']
    bins = [-float('inf'), -8, 0, float('inf')]
    group_names = ['High_Conflict', 'Low_Conflict', 'Cooperation']
    df_copy['relation'] = pd.cut(goldstein, bins, labels=group_names)
    return df_copy


def build_monthly_label(df, high_threshold_quantile=0.9, low_threshold_quantile=0.75):
    df_copy = df.copy()
    df_copy = classify_event(df_copy)
    counts = df_copy.groupby(['dyad', 'MonthYear'])['relation'].value_counts().unstack(fill_value=0)
    counts['total'] = counts['Cooperation'] + counts['Low_Conflict'] + counts['High_Conflict']
    counts['high_conflict_pct'] = counts['High_Conflict'] / counts['total']
    counts['low_conflict_pct'] = counts['Low_Conflict'] / counts['total']
    
    high_threshold = counts['high_conflict_pct'].quantile(high_threshold_quantile)
    low_threshold = counts['low_conflict_pct'].quantile(low_threshold_quantile) # 原本僅頻大小判斷，現在使用門檻限制
    
    counts['monthly_label'] = np.where(
        counts['high_conflict_pct'] >= high_threshold,
        'High_Conflict',
        np.where(
            counts['low_conflict_pct'] >= low_threshold, # 雖然不太可能相等，但依據預警邏輯，我仍將相等的情況判定為低度衝突
            'Low_Conflict',
            'Cooperation'
        )
    )
    return counts

In [ ]:
# test def classify_event(df)
check = classify_event(df)
high_conflict_by_month = check[check['relation'] == 'High_Conflict'].groupby(['dyad', 'MonthYear']).size()
total_dyad_months = check.groupby(['dyad', 'MonthYear']).ngroups
months_with_high_conflict = high_conflict_by_month.shape[0]

print(f'總共有 {total_dyad_months} 個 dyad-月份組合')
print(f'其中 {months_with_high_conflict} 個至少出現過一筆高度衝突事件')
print(f'佔比: {months_with_high_conflict / total_dyad_months:.2%}')


總共有 1578 個 dyad-月份組合
其中 1454 個至少出現過一筆高度衝突事件
佔比: 92.14%


In [ ]:
monthly_total = check.groupby(['dyad', 'MonthYear']).size()
monthly_high = check[check['relation'] == 'High_Conflict'].groupby(['dyad', 'MonthYear']).size()
high_conflict_pct = (monthly_high / monthly_total).fillna(0)
print(high_conflict_pct.describe())
print(high_conflict_pct.quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

count    1578.000000
mean        0.066480
std         0.072648
min         0.000000
25%         0.027043
50%         0.046073
75%         0.076623
max         1.000000
dtype: float64
0.50    0.046073
0.75    0.076623
0.90    0.146305
0.95    0.200254
0.99    0.329823
dtype: float64


np.float64(0.14630475050190442)

### 紀錄01
月lables：合作、低度衝突、高度衝突
原本標籤有兩個路徑：
1. 依據行為分布決定該月為何種型態
2. 只要有高度衝突出現就將該月為高度衝突(原因:因高度衝突屬嚴重的暴力行為)
#### 驗證一下是否可行
1. 先假設規則2：只要當月出現任一高度衝突事件（Goldstein ≤ -8），即整月標記為高度衝突。
2. 實際驗證：12 組 dyad、2015-2025，共 1,578 個 dyad-月份組合， 其中 1,454 個至少出現過一筆高度衝突事件，佔比92.14%。
   → 規則2不可行，幾乎所有月份都會被標記為高度衝突。

改看 high_conflict_pct（當月高度衝突事件數 / 當月總事件數）的分布：
- 中位數 4.6%，75 分位數 7.7%，90 分位數 14.6%，95 分位數 20.0%，99 分位數 33.0%
- 分布明顯右偏：大部分月份佔比集中在 10% 以下，但存在一批佔比異常高的月份

### 決策
以 high_conflict_pct 的 90 分位數（14.6%）作為門檻：
- high_conflict_pct >= 90 分位數 → 標記「高度衝突」
- 其餘月份 → 比較 Cooperation vs Low_Conflict 何者事件數較多，標記為對應類別
選擇 90 分位數而非 75 分位數，理由是希望「高度衝突」類別聚焦在真正顯著偏離常態的月份，避免把「普通偏高」的月份也一併納入。
- 規則2的方法似乎比較適合以天為分析單位

In [ ]:
# test def build_monthly_label(df,  threshold_quantile=0.9)
monthly_labels = build_monthly_label(df)
print(monthly_labels.info())
print(monthly_labels['monthly_label'].value_counts())
print(monthly_labels.groupby('monthly_label')[['Cooperation', 'Low_Conflict', 'High_Conflict', 'total', 'high_conflict_pct']].describe())

<class 'pandas.DataFrame'>
MultiIndex: 1578 entries, ('CHN-JPN', np.int64(201501)) to ('PRK-TWN', np.int64(202512))
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   High_Conflict      1578 non-null   int64  
 1   Low_Conflict       1578 non-null   int64  
 2   Cooperation        1578 non-null   int64  
 3   total              1578 non-null   int64  
 4   high_conflict_pct  1578 non-null   float64
 5   monthly_label      1578 non-null   str    
dtypes: float64(1), int64(4), str(1)
memory usage: 97.6 KB
None
monthly_label
Cooperation      1297
High_Conflict     158
Low_Conflict      123
Name: count, dtype: int64
relation      Cooperation                                                \
                    count         mean          std  min    25%     50%   
monthly_label                                                             
Cooperation        1297.0  1465.600617  1668.013654  1.0  322.0  1013.0  

In [41]:
monthly_labels['low_conflict_pct'] = monthly_labels['Low_Conflict'] / monthly_labels['total']
print(monthly_labels['low_conflict_pct'].describe())
print(monthly_labels['low_conflict_pct'].quantile([0.5, 0.75, 0.9, 0.95]))

count    1578.000000
mean        0.301394
std         0.136012
min         0.000000
25%         0.207800
50%         0.284936
75%         0.381327
max         1.000000
Name: low_conflict_pct, dtype: float64
0.50    0.284936
0.75    0.381327
0.90    0.459389
0.95    0.513772
Name: low_conflict_pct, dtype: float64


In [45]:
monthly_labels = build_monthly_label(df)
print(monthly_labels['monthly_label'].value_counts())

monthly_label
Cooperation      1084
Low_Conflict      336
High_Conflict     158
Name: count, dtype: int64


### label檢視（第一版：僅 High_Conflict 有獨立門檻）
 
執行 build_monthly_label() 後，1,578 個 dyad-月份的標籤分布：
- Cooperation：1,297（82.2%）
- High_Conflict：158（10.0%，符合90分位數門檻設計）
- Low_Conflict：123（7.8%）
**High_Conflict 類別驗證穩定**：high_conflict_pct平均 23.6%，最小值 14.7%（緊貼門檻）、與 Cooperation（平均 4.6%）、Low_Conflict（平均 5.9%）區隔清楚，判定邊界乾淨。
 
**發現問題：Low_Conflict 與 Cooperation 判定方式（比較絕對筆數）有鑑別力的缺陷**
原規則：high_conflict_pct 未達門檻時，比較 Low_Conflict 筆數是否 >= Cooperation 筆數。
問題舉例：若當月總事件 1000 筆，「合作999、低度衝突1」與「合作501、低度衝突499」兩種情況，用筆數比較會被判成同一類，但後者顯然緊張程度高得多，兩者不該同標籤。

### 標籤規則修正（第二版：Low_Conflict 亦採用獨立門檻）
改為與 High_Conflict 相同邏輯，對 low_conflict_pct 另訂獨立門檻，而非與 Cooperation 比大小：
- low_conflict_pct 分布：中位數 28.5%、75 分位數 38.1%、90 分位數 45.9%、95 分位數 51.4%
- 分布相對集中、非長尾（不同於 high_conflict_pct 的右偏型態），故不採用 90 分位數，暫採 75 分位數（38.1%）
最終規則：
```
high_conflict_pct >= 90分位數 → High_Conflict
否則，若 low_conflict_pct >= 75分位數 → Low_Conflict
否則 → Cooperation
```